In [1]:
# Alumno: Omar Asael Martinez Chaparro
# Matrícula: Al07283164
# Proyecto: Restaurante Mar y Luz

import datetime
import os
import queue
import threading
import time


# Tupla inmutable del menú de platillos: (nombre_platillo, precio)
MENU_PLATILLOS = (
    ("Carne asada", 120.0),
    ("Pechuga empanizada", 120.0),
    ("Pancita con patita de res", 110.0),
    ("Menudo de borrego", 130.0),
    ("Caldo de guajolote", 150.0),
    ("Mole de guajolote", 150.0),
    ("Chiles rellenos", 110.0),
    ("Carne de puerco", 110.0),
)

# Archivos .txt base de las mesas para conservar registros entre sesiones
ARCHIVOS_MESAS = [
    "mar_y_luz_mesa1.txt",
    "mar_y_luz_mesa2.txt",
    "mar_y_luz_mesa3.txt",
    "mar_y_luz_mesa4.txt",
    "mar_y_luz_mesa5.txt",
    "mar_y_luz_mesa6.txt",
    "mar_y_luz_mesa7.txt",
    "mar_y_luz_mesa8.txt",
]

# Matriz de opciones del menú principal: [opcion, descripcion]
MATRIZ_MENU = [
    ["1", "Agregar platillo"],
    ["2", "Consultar pedido actual"],
    ["3", "Ver historial guardado de una mesa (.txt)"],
    ["4", "Finalizar cuenta, guardar en tupla/archivo y salir"],
]

LIMITE_INACTIVIDAD_SEGUNDOS = 10 * 60  # 10 minutos


def es_nombre_valido(texto):
    """Verifica que el texto contenga al menos una letra y solo letras o espacios."""
    if not texto:
        return False
    tiene_letra = False
    for caracter in texto:
        if caracter.isalpha():
            tiene_letra = True
        elif not caracter.isspace():
            return False
    return tiene_letra


def solicitar_usuario():
    """Solicita un nombre válido (sin números ni símbolos) y lo devuelve limpio."""
    while True:
        nombre = input("Ingrese su nombre: ").strip()
        if es_nombre_valido(nombre):
            return nombre.title()
        print("Nombre no válido. Ingrese solo letras (sin números ni caracteres especiales).")


def mostrar_bienvenida(usuario):
    """Genera el mensaje de bienvenida usando operadores de string."""
    saludo = "Bienvenido/a, " + usuario + ", al sistema del Restaurante Mar y Luz."
    borde = "=" * len(saludo)
    print("\n" + borde)
    print(saludo)
    print(borde)


def mostrar_carga(segundos=3):
    """Muestra un aviso de carga en pantalla de 3 segundos (máximo 5)."""
    if segundos > 5:
        segundos = 5
    print("\nCargando sistema del Restaurante Mar y Luz...")
    for paso in range(1, segundos + 1):
        print(f"Cargando... {paso}/{segundos} s")
        time.sleep(1)
    print("Sistema listo.\n")


def pedir_entero(mensaje, minimo=None, maximo=None):
    """Solicita un número entero validando límites opcionales."""
    while True:
        try:
            valor = int(input(mensaje))
            if minimo is not None and valor < minimo:
                print(f"El valor debe ser como mínimo {minimo}.")
                continue
            if maximo is not None and valor > maximo:
                print(f"El valor debe ser como máximo {maximo}.")
                continue
            return valor
        except ValueError:
            print("Entrada no válida. Ingrese un número entero.")


def leer_con_ciclo_10_min(mensaje):
    """Mide hasta 10 minutos con un ciclo for para esperar una entrada del usuario."""
    while True:
        respuestas = queue.Queue()

        def leer_entrada():
            try:
                respuestas.put((True, input(mensaje)))
            except Exception as error:
                respuestas.put((False, error))

        hilo = threading.Thread(target=leer_entrada, daemon=True)
        hilo.start()

        entrada_recibida = None
        for _ in range(LIMITE_INACTIVIDAD_SEGUNDOS):
            try:
                ok, dato = respuestas.get(timeout=1)
                if not ok:
                    raise dato
                entrada_recibida = dato
                break
            except queue.Empty:
                continue

        if entrada_recibida is not None:
            return entrada_recibida

        print("\nHan transcurrido 10 minutos sin seleccionar una opción.")
        while True:
            decision = input("¿Desea continuar en el menú? Escriba 'si' o 'no': ").strip().lower()
            if decision == "si":
                print("Continuando en el menú...")
                break
            if decision == "no":
                return None
            print("Respuesta no válida. Escriba 'si' para seguir en el menú o 'no' para ir a la pantalla inicial.")


def seleccionar_opcion_menu():
    """Solicita y valida una opción del 1 al 4 midiendo el tiempo con un ciclo for."""
    while True:
        texto = leer_con_ciclo_10_min("Seleccione una opcion (1-4): ")
        if texto is None:
            return "PANTALLA_INICIAL"
        try:
            valor = int(texto.strip())
            if 1 <= valor <= 4:
                return valor
            print("El valor debe ser como mínimo 1.") if valor < 1 else print("El valor debe ser como máximo 4.")
        except ValueError:
            print("Entrada no válida. Ingrese un número entero.")


def capturar_fecha():
    """Solicita día, mes y año, valida la fecha y regresa la tupla Fecha = dia, mes, anio."""
    print("\nIngrese la fecha de operación (ejemplo: día 12, mes 06, año 2023):")
    while True:
        try:
            dia = pedir_entero("Ingrese el día (1-31): ", 1, 31)
            mes = pedir_entero("Ingrese el mes (1-12): ", 1, 12)
            anio_texto = input("Ingrese el año (4 dígitos): ").strip()
            if len(anio_texto) != 4 or not anio_texto.isdigit():
                print("Año no válido. Debe tener exactamente 4 dígitos.")
                continue
            anio = int(anio_texto)
            datetime.date(anio, mes, dia)
            Fecha = dia, mes, anio
            return Fecha
        except ValueError:
            print("Fecha no válida. Verifique día, mes y año e intente de nuevo.")


def preparar_archivos_mesas():
    """Verifica o crea los archivos .txt de las mesas sin borrar información previa."""
    for nombre in ARCHIVOS_MESAS:
        try:
            if not os.path.exists(nombre):
                with open(nombre, "w", encoding="utf-8") as archivo:
                    archivo.write("HISTORIAL DE PEDIDOS - RESTAURANTE MAR Y LUZ\n")
                    archivo.write("=" * 48 + "\n")
        except PermissionError:
            print(f"No hay permisos para crear '{nombre}'.")
        except OSError as error:
            print(f"Error al preparar '{nombre}': {error}")


def mostrar_historial_al_reiniciar():
    """Muestra un resumen de los registros guardados en sesiones anteriores al iniciar."""
    print("\n--- HISTORIAL PREVIO GUARDADO ---")
    hay_registros = False
    for indice, nombre_archivo in enumerate(ARCHIVOS_MESAS, 1):
        try:
            with open(nombre_archivo, "r", encoding="utf-8") as archivo:
                contenido = archivo.read()
                if "REGISTRO DE CUENTA" in contenido:
                    hay_registros = True
                    print(f"[Mesa {indice}] -> '{nombre_archivo}' contiene registros previos.")
        except FileNotFoundError:
            continue
        except PermissionError:
            print(f"[Mesa {indice}] -> Sin permisos para leer '{nombre_archivo}'.")
        except OSError as error:
            print(f"[Mesa {indice}] -> Error al revisar '{nombre_archivo}': {error}")
    if not hay_registros:
        print("Aún no hay cuentas cerradas en sesiones anteriores.")
    print("-" * 34 + "\n")


def mostrar_menu_platillos():
    """Muestra los platillos almacenados en la tupla MENU_PLATILLOS."""
    print("\nMENU DE PLATILLOS")
    for idx in range(len(MENU_PLATILLOS)):
        nombre, precio = MENU_PLATILLOS[idx]
        print(f"{idx + 1}. {nombre} - ${precio:.2f}")


def agregar_platillo(registros_mesa):
    """Registra un platillo como tupla (nombre, cantidad, precio, importe)."""
    if len(registros_mesa) >= 8:
        print("Se alcanzo el limite de ocho registros.")
        print("Consulte el pedido o finalice la cuenta.")
        return

    mostrar_menu_platillos()
    platillo = pedir_entero("Seleccione el platillo (1-8): ", 1, len(MENU_PLATILLOS))
    cantidad = pedir_entero("Ingrese la cantidad: ", 1)

    nombre, precio = MENU_PLATILLOS[platillo - 1]
    importe = precio * cantidad
    registro = (nombre, cantidad, precio, importe)
    registros_mesa.append(registro)

    print("\nPlatillo registrado correctamente.")
    print(f"Importe del platillo: ${importe:.2f}")
    print("Una vez enviado a cocina, no se puede modificar.")


def calcular_total(registros_mesa):
    """Calcula y devuelve el total acumulado de una lista de tuplas."""
    total = 0.0
    for item in registros_mesa:
        _, _, _, importe = item
        total = total + importe
    return total


def consultar_pedido(mesa, registros_mesa):
    """Muestra los platillos registrados en la mesa y el total actual."""
    print("\n================================")
    print(f"       PEDIDO DE LA MESA {mesa}")
    print("================================")
    if len(registros_mesa) == 0:
        print("No hay platillos registrados.")
        return

    for i in range(len(registros_mesa)):
        nombre, cantidad, precio, importe = registros_mesa[i]
        print(
            f"Registro {i + 1} | {nombre} | Cantidad: {cantidad} "
            f"| Precio: ${precio:.2f} | Importe: ${importe:.2f}"
        )
    total = calcular_total(registros_mesa)
    print("--------------------------------")
    print(f"TOTAL ACTUAL: ${total:.2f}")


def procesar_pago_separado(registros_mesa, total):
    """Divide la cuenta por persona y devuelve las líneas del resumen."""
    print("\n================================")
    print("      PAGO SEPARADO")
    print("================================")
    print("Cada persona pagara los platillos que consumio.\n")
    print("REGISTROS DISPONIBLES")
    for i in range(len(registros_mesa)):
        nombre, cantidad, _, importe = registros_mesa[i]
        print(f"Registro {i + 1} | {nombre} | Cantidad: {cantidad} | Importe: ${importe:.2f}")

    personas = pedir_entero(
        "\n¿Cuantas personas pagaran por separado? ", 1, len(registros_mesa)
    )

    while True:
        registros_asignados = []
        pagos = []

        for persona in range(1, personas + 1):
            while True:
                entrada = input(
                    f"\nPersona {persona}, Ingrese el numero de pedido que consumiste separado con comas: "
                )
                try:
                    registros = []
                    for dato in entrada.split(","):
                        registros.append(int(dato.strip()))

                    if len(registros) == 0:
                        print("Debe ingresar al menos un registro.")
                        continue
                    if len(set(registros)) != len(registros):
                        print("No repita el mismo registro.")
                        continue

                    validos = True
                    for numero in registros:
                        if numero < 1 or numero > len(registros_mesa):
                            validos = False
                    if not validos:
                        print("Uno de los registros no existe.")
                        continue

                    repetido = False
                    for numero in registros:
                        if numero in registros_asignados:
                            repetido = True
                    if repetido:
                        print("Un registro ya fue asignado a otra persona.")
                        continue

                    break
                except ValueError:
                    print("Ingrese el numero de pedido que consumiste separado con comas. Ejemplo: 1,3")

            subtotal = 0.0
            for numero in registros:
                _, _, _, importe = registros_mesa[numero - 1]
                subtotal = subtotal + importe

            registros_asignados.extend(registros)
            pagos.append(subtotal)

        if len(registros_asignados) == len(registros_mesa):
            break
        print("\nFaltan registros por asignar.")
        print("Se solicitara nuevamente la distribución.")
        print("Cada platillo debe asignarse una sola vez.")

    resumen = []
    print("\n================================")
    print("       RESUMEN DE PAGOS")
    print("================================")
    total_pagos = 0.0
    for idx in range(personas):
        linea = f"Persona {idx + 1} debe pagar: ${pagos[idx]:.2f}"
        print(linea)
        resumen.append(linea)
        total_pagos = total_pagos + pagos[idx]

    print("--------------------------------")
    print(f"Total de la mesa: ${total:.2f}")
    print(f"Suma de pagos individuales: ${total_pagos:.2f}")
    if round(total_pagos, 2) == round(total, 2):
        estado = "Comprobacion correcta: los pagos coinciden con la cuenta."
    else:
        estado = "Error: los pagos no coinciden con el total."
    print(estado)
    resumen.append(estado)
    return resumen


def guardar_cuenta_en_archivo(usuario, mesa, Fecha, registros_mesa, tipo_pago, resumen_pago):
    """Guarda los registros en el archivo .txt de la mesa usando Fecha y una tupla de cierre."""
    dia, mes, anio = Fecha
    total = calcular_total(registros_mesa)
    # Tupla que agrupa los datos principales guardados al cerrar el programa
    registro_cierre = (dia, mes, anio, mesa, usuario, tipo_pago, total)
    d_val, m_val, a_val, m_num, usr_val, pago_val, tot_val = registro_cierre
    nombre_archivo = ARCHIVOS_MESAS[m_num - 1]

    try:
        with open(nombre_archivo, "a", encoding="utf-8") as archivo:
            archivo.write("\nREGISTRO DE CUENTA - RESTAURANTE MAR Y LUZ\n")
            archivo.write(f"Fecha (día, mes, año): {d_val, m_val, a_val}\n")
            archivo.write(f"Mesa: {m_num} | Atendió: {usr_val} | Modalidad: {pago_val}\n")
            for idx in range(len(registros_mesa)):
                nombre, cantidad, precio, importe = registros_mesa[idx]
                archivo.write(
                    f"Registro {idx + 1} | {nombre} | Cantidad: {cantidad} "
                    f"| Precio: ${precio:.2f} | Importe: ${importe:.2f}\n"
                )
            for linea in resumen_pago:
                archivo.write(f"{linea}\n")
            archivo.write(f"TOTAL FINAL: ${tot_val:.2f}\n")
            archivo.write("-" * 48 + "\n")
        print(f"\nRegistros guardados correctamente en '{nombre_archivo}'.")
    except FileNotFoundError:
        print(f"\nNo se encontró el archivo '{nombre_archivo}'.")
    except PermissionError:
        print(f"\nNo hay permisos para escribir en '{nombre_archivo}'.")
    except OSError as error:
        print(f"\nError al guardar en '{nombre_archivo}': {error}")


def leer_archivo_mesa():
    """Permite seleccionar una mesa, leer su archivo .txt y decidir si seguir o poner fin."""
    print("\n========== ARCHIVOS DE MESAS ==========")
    for numero, nombre in enumerate(ARCHIVOS_MESAS, 1):
        print(f"{numero}. {nombre}")

    opcion = pedir_entero("Seleccione el número de mesa para leer su archivo (1-8): ", 1, 8)
    nombre_archivo = ARCHIVOS_MESAS[opcion - 1]

    try:
        with open(nombre_archivo, "r", encoding="utf-8") as archivo:
            print(f"\n--- CONTENIDO DE {nombre_archivo} ---")
            print(archivo.read())
    except FileNotFoundError:
        print(f"El archivo '{nombre_archivo}' no existe.")
    except PermissionError:
        print(f"No hay permisos para leer '{nombre_archivo}'.")
    except OSError as error:
        print(f"Error al leer '{nombre_archivo}': {error}")

    while True:
        decision = input("\n¿Desea seguir con el programa o poner fin? (seguir/fin): ").strip().lower()
        if decision == "seguir":
            return True
        elif decision == "fin":
            return False
        else:
            print("Respuesta no válida. Escriba 'seguir' para continuar o 'fin' para terminar.")


def ejecutar_programa():
    preparar_archivos_mesas()

    while True:
        usuario = solicitar_usuario()
        mostrar_bienvenida(usuario)
        mostrar_carga(segundos=3)
        mostrar_historial_al_reiniciar()

        Fecha = capturar_fecha()
        mesa = pedir_entero("Ingrese el numero de mesa (1 al 8): ", 1, 8)

        # Diccionario de comandas por mesa
        comandas_mesas = {mesa: []}
        registros_mesa = comandas_mesas[mesa]

        volver_a_inicio = False
        cuenta_finalizada = False

        while True:
            print("\nMENU PRINCIPAL")
            for fila in MATRIZ_MENU:
                print(f"{fila[0]}. {fila[1]}")

            opcion = seleccionar_opcion_menu()

            if opcion == "PANTALLA_INICIAL":
                volver_a_inicio = True
                break
            elif opcion == 1:
                agregar_platillo(registros_mesa)
            elif opcion == 2:
                consultar_pedido(mesa, registros_mesa)
            elif opcion == 3:
                continuar = leer_archivo_mesa()
                if not continuar:
                    cuenta_finalizada = True
                    break
            elif opcion == 4:
                if len(registros_mesa) == 0:
                    print("No hay platillos registrados.")
                    print("Agregue al menos un platillo antes de finalizar.")
                    continue

                total = calcular_total(registros_mesa)
                print("\n================================")
                print("          CUENTA FINAL")
                print("================================")
                print(f"Mesa: {mesa}")
                for i in range(len(registros_mesa)):
                    nombre, cantidad, precio, importe = registros_mesa[i]
                    print(
                        f"Registro {i + 1} | {nombre} | Cantidad: {cantidad} "
                        f"| Precio: ${precio:.2f} | Importe: ${importe:.2f}"
                    )
                print("--------------------------------")
                print(f"TOTAL DE LA MESA: ${total:.2f}")

                print("\n¿COMO DESEA PAGAR?")
                print("1. Pago junto")
                print("2. Pago separado por platillo")
                forma_pago = pedir_entero("Seleccione 1 o 2: ", 1, 2)

                if forma_pago == 1:
                    tipo_pago = "Pago junto"
                    resumen_pago = [f"TOTAL A PAGAR: ${total:.2f}"]
                    print("\n================================")
                    print("             PAGO JUNTO")
                    print("================================")
                    print("Una sola cuenta para toda la mesa.")
                    print(f"TOTAL A PAGAR: ${total:.2f}")
                else:
                    tipo_pago = "Pago separado"
                    resumen_pago = procesar_pago_separado(registros_mesa, total)

                guardar_cuenta_en_archivo(usuario, mesa, Fecha, registros_mesa, tipo_pago, resumen_pago)
                cuenta_finalizada = True
                break

        if volver_a_inicio:
            print("\nRegresando a la pantalla inicial...")
            continue

        if cuenta_finalizada:
            break

    print("\nGracias por visitar el Restaurante Mar y Luz.")
    print("¡Vuelva pronto!")


if __name__ == "__main__":
    ejecutar_programa()

Ingrese su nombre: moises

Bienvenido/a, Moises, al sistema del Restaurante Mar y Luz.

Cargando sistema del Restaurante Mar y Luz...
Cargando... 1/3 s
Cargando... 2/3 s
Cargando... 3/3 s
Sistema listo.


--- HISTORIAL PREVIO GUARDADO ---
Aún no hay cuentas cerradas en sesiones anteriores.
----------------------------------


Ingrese la fecha de operación (ejemplo: día 12, mes 06, año 2023):
Ingrese el día (1-31): 28
Ingrese el mes (1-12): 11
Ingrese el año (4 dígitos): 2026
Ingrese el numero de mesa (1 al 8): 1

MENU PRINCIPAL
1. Agregar platillo
2. Consultar pedido actual
3. Ver historial guardado de una mesa (.txt)
4. Finalizar cuenta, guardar en tupla/archivo y salir
Seleccione una opcion (1-4): 1

MENU DE PLATILLOS
1. Carne asada - $120.00
2. Pechuga empanizada - $120.00
3. Pancita con patita de res - $110.00
4. Menudo de borrego - $130.00
5. Caldo de guajolote - $150.00
6. Mole de guajolote - $150.00
7. Chiles rellenos - $110.00
8. Carne de puerco - $110.00
Seleccione el platillo